In [0]:
# This notebook copies the Genie Code skills from this repo into your personal skills folder.
# Run it once per workspace after deploying or cloning the project, before you start adapting the template with Genie Code.

In [0]:
import os


def to_file_uri(path: str) -> str:
    return path if path.startswith("file:") else f"file:{path}"


def workspace_path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(to_file_uri(path))
        return True
    except Exception:
        return False


def resolve_source_skills_path() -> str:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    candidates = []

    if "/files/notebooks/" in notebook_path:
        files_root = notebook_path.split("/files/notebooks/")[0] + "/files"
        candidates.append(os.path.join(files_root, ".assistant", "skills"))

    if "/notebooks/" in notebook_path:
        repo_root = notebook_path.split("/notebooks/")[0]
        candidates.append(os.path.join(repo_root, ".assistant", "skills"))

    workspace_candidates = [
        f"/Workspace{path}" for path in candidates if not path.startswith("/Workspace")
    ]
    candidates.extend(workspace_candidates)

    for path in candidates:
        if workspace_path_exists(path):
            return path

    raise ValueError(
        "Could not find .assistant/skills in this workspace. "
        f"Tried: {candidates}"
    )


username = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
source_skills_path = resolve_source_skills_path()
dest_skills_path = f"/Workspace/Users/{username}/.assistant/skills"

dbutils.fs.mkdirs(to_file_uri(f"/Workspace/Users/{username}/.assistant"))
dbutils.fs.mkdirs(to_file_uri(dest_skills_path))

copied_skills = []
for item in dbutils.fs.ls(to_file_uri(source_skills_path)):
    if not item.isDir():
        continue

    skill_name = item.name.rstrip("/")
    source_skill_path = f"{to_file_uri(source_skills_path)}/{skill_name}"
    dest_skill_path = f"{to_file_uri(dest_skills_path)}/{skill_name}"

    dbutils.fs.cp(source_skill_path, dest_skill_path, recurse=True)
    copied_skills.append(skill_name)

print(f"Source: {source_skills_path}")
print(f"Destination: {dest_skills_path}")
print(f"Copied {len(copied_skills)} skills:")
for skill_name in sorted(copied_skills):
    print(f"  - {skill_name}")